# Kiểm thử chatbot v2 (tài khoản leader, dữ liệu thật 26 sản phẩm)
Chạy từ trên xuống. Các ô 2-5 chỉ đọc dữ liệu hoặc không ghi gì. **Ô 6 ghi giỏ hàng vào bảng thật** bằng user `TEST-NOTEBOOK`;
sau khi chạy xong, xóa giỏ này trong CloudShell:
`aws dynamodb delete-item --table-name Carts --key '{"CartID":{"S":"TEST-NOTEBOOK"}}' --region ap-southeast-1`

In [ ]:
# Ô 1: thư viện, địa chỉ và hàm hỏi
!pip install -q websocket-client
import json, time, uuid
import websocket

URL = "wss://eu9j8c0avi.execute-api.ap-southeast-1.amazonaws.com/dev/"
GARAGE_USER = "USR_98765"      # user có xe Exciter 150 (2019) trong Garage; chỉ dùng để ĐỌC, không thêm giỏ
CART_USER = "TEST-NOTEBOOK"    # user riêng cho test giỏ hàng (xóa sau khi chạy)

def ask(question, session_id, user_id=None, timeout=180, quiet=False):
    ws = websocket.create_connection(URL, timeout=timeout)
    payload = {"action": "sendMessage", "question": question, "session_id": session_id}
    if user_id:
        payload["user_id"] = user_id
    t = time.time()
    ws.send(json.dumps(payload))
    try:
        raw = ws.recv()
    finally:
        ws.close()
    took = time.time() - t
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        print("Phản hồi không phải JSON:", raw); return None
    data["_took"] = took
    if not quiet:
        print(f"\n>>> {question}   ({took:.1f}s)")
        print("Bot:", data.get("answer"))
        for p in data.get("products") or []:
            print(f"  - {p['ProductID']} | {p['name']} | {p['price']} VND | tồn kho: {p['stock']}")
        if data.get("sources"): print("Nguồn:", data["sources"])
        print("low_confidence:", data.get("low_confidence"), "| cart_updated:", data.get("cart_updated"), "| ticket:", data.get("handover_ticket"))
    return data

print(URL)

## Ô 2: Làm nóng (câu đầu sau khi Lambda nghỉ có thể mất ~7 giây)

In [ ]:
ask("Chính sách đổi trả thế nào?", "warm-" + uuid.uuid4().hex[:6])

## Ô 3: Chính sách (kiểm tra tài liệu KB)

In [ ]:
KB_TESTS = [  # (câu hỏi, các từ khóa PHẢI có trong câu trả lời - chỉ cần 1, nguồn mong đợi)
    ("Phụ tùng được bảo hành bao lâu?", ["6 tháng"], "chinh_sach_bao_hanh.txt"),
    ("Đổi trả trong bao nhiêu ngày?", ["7 ngày"], "chinh_sach_doi_tra.txt"),
    ("Nhớt có được bảo hành không?", ["7 ngày", "lỗi sản xuất", "tiêu hao"], "chinh_sach_bao_hanh.txt"),
    ("Đơn hàng bao nhiêu thì miễn phí vận chuyển?", ["miễn phí"], "chinh_sach_van_chuyen.txt"),
    ("Xe côn tay nên dùng nhớt loại nào?", ["JASO", "MA"], "huong_dan_chon_phu_tung.txt"),
    ("Dấu hiệu phuộc bị hỏng là gì?", ["rò", "dầu", "nhún"], "huong_dan_chon_phu_tung.txt"),
]
sid = "kb-" + uuid.uuid4().hex[:6]
kb_results = []
for q, kws, src in KB_TESTS:
    d = ask(q, sid)
    ans = (d or {}).get("answer", "") or ""
    ok = bool(d) and not d.get("low_confidence") and not d.get("degraded") and any(k.lower() in ans.lower() for k in kws)
    kb_results.append((ok, q, d["_took"] if d else None))
print("\n===== KẾT QUẢ KB =====")
for ok, q, t in kb_results: print("ĐẠT " if ok else "CHƯA", f"| {t:.1f}s |", q)

## Ô 4: Tư vấn sản phẩm theo dữ liệu thật (không ghi dữ liệu)

In [ ]:
PRODUCT_TESTS = [  # (câu hỏi, ProductID mong đợi - chỉ cần 1, ProductID không được xuất hiện)
    ("Bố thắng cho Winner X đời 2020", {"PROD_BRAKE_001"}, set()),
    ("Bố thắng đĩa trước cho Exciter 150 đời 2019", {"PROD_BRK_001"}, set()),
    ("Tay thắng cùm dầu RCB giá bao nhiêu?", {"PROD_BRAKE_003"}, set()),
    ("Lọc gió K&N còn hàng không?", {"PROD_FILTER_001"}, set()),
    ("Bugi cho Winner X", {"PROD_SPARK_001", "PROD_SPARK_002"}, set()),
    ("Phuộc sau cho Wave Alpha", {"PROD_SUSP_002"}, set()),
    ("Phuộc sau cho Vision", {"PROD_SUSP_001", "PROD_SHOCK_002"}, set()),
    ("Lốp không săm cho AirBlade", {"PROD_TYRE_001"}, set()),
    ("Ắc quy cho Sirius", {"PROD_BATT_001"}, set()),
    ("Pô độ Yoshimura", {"PROD_EXHAUST_001"}, set()),
    ("Gương gù tay lái mọi dòng xe", {"PROD_ACC_001"}, set()),
    ("Sên nhông dĩa cho Raider 150", {"PROD_DRIVE_001", "PROD_DRV_002"}, set()),
]
pr = []
for q, want, ban in PRODUCT_TESTS:
    d = ask(q, "prod-" + uuid.uuid4().hex[:6])  # mỗi câu một phiên riêng: giới hạn 8 câu/phút/phiên của chatbot
    ids = {p["ProductID"] for p in (d or {}).get("products") or []}
    pr.append((bool(want & ids) and not (ban & ids) and not (d or {}).get("degraded"), q, d["_took"] if d else None))
print("\n===== KẾT QUẢ SẢN PHẨM =====")
for ok, q, t in pr: print("ĐẠT " if ok else "CHƯA", f"| {t:.1f}s |", q)

print("\n--- Sản phẩm không có trong kho: bot không được bịa ---")
d = ask("Phuộc trước cho Exciter 150 còn hàng không?", "prod-" + uuid.uuid4().hex[:6])

## Ô 5: Câu ngoài phạm vi và Garage (không ghi dữ liệu, trừ phiếu chuyển nhân viên nếu có)

In [ ]:
OOS = ["Hôm nay thời tiết ở Hà Nội thế nào?", "Viết cho tôi bài thơ về mùa thu", "Giá bitcoin hôm nay", "Cách nấu phở bò"]
oos = []
for q in OOS:
    d = ask(q, "oos-" + uuid.uuid4().hex[:6])
    oos.append((bool(d) and not d.get("products") and not d.get("handover_ticket"), q, d["_took"] if d else None))
print("\n===== KẾT QUẢ NGOÀI PHẠM VI (không gợi ý sản phẩm, không tạo phiếu) =====")
for ok, q, t in oos: print("ĐẠT " if ok else "CHƯA", f"| {t:.1f}s |", q)

print("\n--- Garage: user có xe lưu sẵn ---")
sid = "garage-" + uuid.uuid4().hex[:6]
ask("Có bố thắng nào lắp vừa xe của tôi không?", sid, GARAGE_USER)

## Ô 6: Giỏ hàng (GHI vào bảng Carts thật bằng user TEST-NOTEBOOK - nhớ xóa sau)

In [ ]:
sid = "cart-" + uuid.uuid4().hex[:6]
print("--- Khách chưa đăng nhập: không được dùng giỏ hàng ---")
ask("Thêm 1 tay thắng cùm dầu RCB S1 vào giỏ", "guest-" + uuid.uuid4().hex[:6])
print("\n--- Thêm 2 cái ---")
d1 = ask("Xe của mình là Yamaha Exciter 150 đời 2019. Thêm 2 tay thắng cùm dầu RCB S1 vào giỏ hàng", sid, CART_USER)
print("\n--- Vượt tồn kho: bot phải từ chối và hỏi lại, KHÔNG tự thêm số khác ---")
d2 = ask("Thêm 50 tay thắng cùm dầu RCB S1 vào giỏ (xe Exciter 150 đời 2019)", sid, CART_USER)
print("\n--- Xem giỏ: phải vẫn là 2 sản phẩm ---")
d3 = ask("Xem giỏ hàng của tôi", sid, CART_USER)
cart_ok = bool(d1 and d1.get("cart_updated")) and bool(d2) and not d2.get("cart_updated") and "2" in (d3.get("answer") or "")
print("\nGiỏ hàng:", "ĐẠT" if cart_ok else "CHƯA ĐẠT")

## Ô 7: Chuyển nhân viên (tạo phiếu và gửi email qua Gmail SMTP)

In [ ]:
d = ask("Tôi muốn gặp nhân viên, đơn hàng của tôi có vấn đề", "ho-" + uuid.uuid4().hex[:6], GARAGE_USER, timeout=300)
print("\nChuyển nhân viên:", "ĐẠT" if d and d.get("handover_ticket") else "CHƯA ĐẠT", "(kiểm tra hộp thư Gmail có thư [Chatbot] Khach can ho tro)")

## Ô 8: Tổng hợp

In [ ]:
allr = kb_results + pr + oos
times = [t for _, _, t in allr if t]
print(f"Đạt {sum(1 for ok,_,_ in allr if ok)}/{len(allr)} câu tự động")
print(f"Thời gian trung bình {sum(times)/len(times):.1f}s | chậm nhất {max(times):.1f}s | nhanh nhất {min(times):.1f}s")
print("Giỏ hàng:", "ĐẠT" if cart_ok else "CHƯA ĐẠT")